# 07. Official-test evaluation and paired comparisons

Loads any trained run, generates answers for the 108 official test texts (resumable), and compares systems with
paired bootstrap tests on **all 108** and the **clean 91** (excluding texts in the teacher's prompt pool).

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes

In [ ]:
import os, json, re
import numpy as np, torch
from google.colab import userdata, drive
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

login(token=userdata.get("HF_TOKEN"))
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
if not os.path.exists("/content/maintie"):
    os.system("git clone -q https://github.com/nlp-tlp/maintie.git /content/maintie")
BASE = "/content/drive/MyDrive/mistral_ft"
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
RESULTS_DIR = f"{BASE}/results"; os.makedirs(RESULTS_DIR, exist_ok=True)
gold = json.load(open("/content/maintie/data/gold_release.json"))
splits = json.load(open(f"{BASE}/data/splits.json"))
official = json.load(open(f"{BASE}/data/official_split.json"))["official_test"]
pool = set(splits["excluded_prompt_pool"])
INSTRUCTION = ("Extract the entities (with their MaintIE types) and the relations from this maintenance "
               "work order. Answer with JSON only.\nWork order: ")

def load_trained_model(run_name):
    adapter = f"{BASE}/outputs/{run_name}/final_adapter"
    gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    quant = None if gpu_gb >= 30 else BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    tok.pad_token = tok.pad_token or tok.eos_token; tok.padding_side = "left"
    m = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=quant, device_map="auto",
                                             torch_dtype=torch.bfloat16 if quant is None else None)
    m = PeftModel.from_pretrained(m, adapter)
    if quant is None:
        m = m.merge_and_unload()
    return tok, m.eval()

def generate_predictions(tok, m, indices, out_file, batch=32):
    done = {}
    if os.path.exists(out_file):
        done = {d["gold_idx"]: d["raw"] for d in (json.loads(l) for l in open(out_file) if l.strip())}
    todo = [i for i in indices if i not in done]
    for b in range(0, len(todo), batch):
        chunk = todo[b:b + batch]
        prompts = [tok.apply_chat_template([{"role": "user", "content": INSTRUCTION + gold[i]["text"]}],
                                           add_generation_prompt=True, tokenize=False) for i in chunk]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(m.device)
        with torch.no_grad():
            out = m.generate(**enc, max_new_tokens=512, do_sample=False, pad_token_id=tok.pad_token_id)
        with open(out_file, "a") as f:
            for i, raw in zip(chunk, tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)):
                f.write(json.dumps({"gold_idx": i, "raw": raw}) + "\n")
    return {d["gold_idx"]: d["raw"] for d in (json.loads(l) for l in open(out_file) if l.strip())}

def parse_json(raw):
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", (raw or "").strip())
    s, e = text.find("{"), text.rfind("}")
    try:
        return json.loads(text[s:e + 1]) if s != -1 and e != -1 else None
    except json.JSONDecodeError:
        return None

def locate(tokens, words, used):
    for st in range(len(tokens) - len(words) + 1):
        if tokens[st:st + len(words)] == words and (st, st + len(words)) not in used:
            return (st, st + len(words))
    return None

def score_text(rec, pred):
    tokens = rec["tokens"]
    g_ents = [((e["start"], e["end"]), e["type"]) for e in rec["entities"]]
    g_rels = {(g_ents[r["head"]][0], g_ents[r["tail"]][0], r["type"]) for r in rec["relations"]}
    g_rels_t = {(g_ents[r["head"]], g_ents[r["tail"]], r["type"]) for r in rec["relations"]}
    p_ents, used = [], set()
    for e in (pred or {}).get("entities", []) if isinstance(pred, dict) else []:
        words = str(e.get("text", "")).split()
        span = locate(tokens, words, used) if words else None
        if span:
            used.add(span)
        p_ents.append((span, str(e.get("type", ""))))
    p_rels, p_rels_t = set(), set()
    for r in (pred or {}).get("relations", []) if isinstance(pred, dict) else []:
        h, t = r.get("head"), r.get("tail")
        if isinstance(h, int) and isinstance(t, int) and 0 <= h < len(p_ents) and 0 <= t < len(p_ents):
            if p_ents[h][0] and p_ents[t][0]:
                p_rels.add((p_ents[h][0], p_ents[t][0], r.get("type")))
                p_rels_t.add((p_ents[h], p_ents[t], r.get("type")))
    G = set(g_ents); P = {x for x in p_ents if x[0] is not None}
    unplaced = sum(1 for x in p_ents if x[0] is None)
    return {"entity_exact": (len(G & P), len(P - G) + unplaced, len(G - P)),
            "relation": (len(g_rels & p_rels), len(p_rels - g_rels), len(g_rels - p_rels)),
            "relation_strict": (len(g_rels_t & p_rels_t), len(p_rels_t - g_rels_t), len(g_rels_t - p_rels_t))}

def prf(tp, fp, fn):
    p = tp / max(1, tp + fp); r = tp / max(1, tp + fn)
    return p, r, 2 * tp / max(1, 2 * tp + fp + fn)

## Generate official-test predictions for each run (one GPU session per run is fine)

In [ ]:
for RUN in ["distilled_qwen_nothink_v1", "gold_official_v1", "distilled_860_v1"]:
    out_file = f"{RESULTS_DIR}/predictions_{RUN}_official.jsonl"
    tok, model = load_trained_model(RUN)
    generate_predictions(tok, model, official, out_file)
    del model; torch.cuda.empty_cache()
    print("done", RUN)

## Paired comparisons (bootstrap; diff = A − B)

In [ ]:
def load_preds(path):
    return {d["gold_idx"]: d["raw"] for d in (json.loads(l) for l in open(path) if l.strip())}

def entity_sets(rec, q):
    tokens, used, P = rec["tokens"], set(), []
    for e in (q or {}).get("entities", []) if isinstance(q, dict) else []:
        span = locate(tokens, str(e.get("text", "")).split(), used)
        if span:
            used.add(span); P.append((span, str(e.get("type", ""))))
    return {((e["start"], e["end"]), e["type"]) for e in rec["entities"]}, set(P)

def per_text_type_counts(preds, idx):
    out = []
    for i in idx:
        Gs, Ps = entity_sets(gold[i], parse_json(preds[i]))
        d = {}
        for t in {x[1] for x in Gs | Ps}:
            Gt, Pt = {x for x in Gs if x[1] == t}, {x for x in Ps if x[1] == t}
            d[t] = (len(Gt & Pt), len(Pt - Gt), len(Gt - Pt))
        out.append(d)
    return out

def macro_from(per_text, sel):
    tot = {}
    for k in sel:
        for t, c in per_text[k].items():
            a = tot.setdefault(t, [0, 0, 0]); a[0] += c[0]; a[1] += c[1]; a[2] += c[2]
    return np.mean([prf(*c)[2] for c in tot.values()])

SYSTEMS = {"Distilled 6,709": "distilled_qwen_nothink_v1", "Distilled 860": "distilled_860_v1", "Gold 860": "gold_official_v1"}
preds = {n: load_preds(f"{RESULTS_DIR}/predictions_{r}_official.jsonl") for n, r in SYSTEMS.items()}
METRICS = {"entity_exact": "Entity micro F1", "relation": "Relation, spans only", "relation_strict": "Relation, with types"}
SUBSETS = {"All 108": official, "Clean 91": [i for i in official if i not in pool]}

def compare(a, b, idx, n_boot=2000):
    rng = np.random.default_rng(0)
    samples = [rng.integers(0, len(idx), len(idx)) for _ in range(n_boot)]
    print(f"\n  {a}  vs  {b}")
    for m, nice in METRICS.items():
        A = np.array([score_text(gold[i], parse_json(preds[a][i]))[m] for i in idx])
        B = np.array([score_text(gold[i], parse_json(preds[b][i]))[m] for i in idx])
        d = np.array([prf(*A[s].sum(0))[2] - prf(*B[s].sum(0))[2] for s in samples])
        fa, fb = prf(*A.sum(0))[2], prf(*B.sum(0))[2]
        p = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())); lo, hi = np.percentile(d, [2.5, 97.5])
        print(f"    {nice:24s} {fa*100:5.1f} vs {fb*100:5.1f}  diff {(fa-fb)*100:+5.1f}  CI {lo*100:+5.1f} to {hi*100:+5.1f}  p={p:.3f}")
    TA, TB = per_text_type_counts(preds[a], idx), per_text_type_counts(preds[b], idx)
    d = np.array([macro_from(TA, s) - macro_from(TB, s) for s in samples])
    ma, mb = macro_from(TA, range(len(idx))), macro_from(TB, range(len(idx)))
    p = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())); lo, hi = np.percentile(d, [2.5, 97.5])
    print(f"    {'Entity macro F1':24s} {ma*100:5.1f} vs {mb*100:5.1f}  diff {(ma-mb)*100:+5.1f}  CI {lo*100:+5.1f} to {hi*100:+5.1f}  p={p:.3f}")

for subset, idx in SUBSETS.items():
    print(f"\n===== {subset} official test texts =====")
    compare("Distilled 860", "Gold 860", idx)
    compare("Distilled 6,709", "Distilled 860", idx)
    compare("Distilled 6,709", "Gold 860", idx)